In [8]:
import OpenMM2gmx
import nglview as nv
import MDAnalysis as mda

This notebook provides an example of applying our tools to center a **Class B GPCR structure bound to a peptide ligand**, simulated using **OpenMM** in a triclinic box.

The complex was not centered in the PBC box. To resolve this, we employ the **OpenMM2gmx** utility to perform centering based on the residue closest to the center of mass (COM) of the protein (default behavior).


* The input files are located in the `GPCR_case` folder and include the topology file (`parm7`), the trajectory (`dcd`), and the OpenMM system  (`xml`) corresponding to the non-centered simulation.

* The centered results are stored in the `MD_output` folder, which contains the processed structure (`gro` file) and the centered trajectory in `xtc` format, allowing direct comparison before and after correction.

## Visualization Before Centering

In [2]:
u = mda.Universe("GPCR_case/topology.parm7", "GPCR_case/traj.dcd")
view = nv.show_mdanalysis(u)
view.clear_representations()
# Protein
view.add_cartoon("protein")
# Water
view.add_licorice("water")
# Ions
view.add_licorice("ions")
# Lipids
view.add_ball_and_stick("not protein and not water and not ions")

view.center()
view

NGLWidget(max_frame=250)

## System Centering

In [3]:
OpenMM2gmx.center_trajs(
    top_file= "GPCR_case/topology.parm7",
    traj_files = "GPCR_case/traj.dcd",
    xml_file= "GPCR_case/traj.xml",
    mdp_file = "example_file.mdp",
    sim_name="GPCR_centered",
    center_res="COM"
)

- Generating .gro and .top files
- Detected box vectors: [104.0937021  104.0937021  139.30341273  90.          90.
 120.        ]
- Conversion completed successfully, Generated files: MD_output/GPCR_centered.gro, MD_output/GPCR_centered.top
- Generating TPR file using GROMACS
- TPR file generated: MD_output/GPCR_centered.tpr
- Converting trajectory files to XTC format


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 251/251 [00:01<00:00, 173.76it/s]

- Trajectory conversion completed: GPCR_centered_trj.xtc
- Generating index file 


- Center of mass: [113.74591912 110.91736451 127.86461703]
- The residue closest to the center of mass is : LEU (Resid: 421)
Running command: gmx make_ndx -f MD_output/GPCR_centered.gro -o MD_output/GPCR_centered_index.ndx
- Index file generated: GPCR_centered_index.ndx with group COM (group id: 21)
- Correcting trajectory for PBC and centering
Running command: gmx trjconv -s MD_output/GPCR_centered.tpr -f MD_output/GPCR_centered_trj.xtc -o MD_output/GPCR_centered_nojump.xtc -pbc nojump
- Trajectory correction completed ! 


## Visualization After Centering

In [6]:
u = mda.Universe("MD_output/GPCR_centered.gro", "MD_output/GPCR_centered_trj_no_jump_centered_full_complex.xtc")
view = nv.show_mdanalysis(u)
view.clear_representations()
# Protein
view.add_cartoon("protein")
# Water
view.add_licorice("water")
# Ions
view.add_licorice("ions")
# Lipids
view.add_ball_and_stick("not protein and not water and not ions")

view.center()
view

NGLWidget(max_frame=250)